### Introduction to Machine Learning
Jesús Urtasun Elizari: MRC LMS 2026

**Chapter 1: What is learning. Exercise 2**

**Supervised learning: classification**

We want to classify small black and white images as **cat** or **dog**. A computer sees an image as a grid of numbers, which we can write as one long vector: this is the **input**.
A **model** turns the input into an **output**: one number per class, how much the image looks like a cat and like a dog. We compare the output with the **label** (the true class) using a **loss**.

**The maths**

For every entry $i$ we have a label $y_i$, the true value, and a prediction $y_i^{\text{pred}}$, the output of the model. The loss is the sum of the squared differences between them:

$$L = \sum_{i} \left( y_i - y_i^{\text{pred}} \right)^2$$

The smaller the loss, the closer the predictions are to the labels.

Here $i$ runs over the 2 classes, cat and dog. The label is a one-hot vector: $y = (1, 0)$ for a cat and $y = (0, 1)$ for a dog. The prediction compares the 256 pixels of the image with a typical cat and a typical dog: on the pixels where the two are different, we count how many agree with the cat and how many with the dog, and divide by the total so the two numbers add up to 1:

$$y^{\text{pred}} = \frac{(\text{agree}_{\text{cat}},\; \text{agree}_{\text{dog}})}{\text{agree}_{\text{cat}} + \text{agree}_{\text{dog}}}$$

For example, for a cat with prediction $(0.8, 0.2)$: $L = (1 - 0.8)^2 + (0 - 0.2)^2 = 0.08$.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt

# Check installation
print("All libraries imported successfully")
print(np.__version__)

In [ ]:
# A typical cat and a typical dog, drawn by hand on a grid of 16 x 16 pixels
# '#' is a black pixel, '.' is a white pixel
cat_drawing = [
    "##............##",   # pointy ears
    "###..........###",
    "####........####",
    "################",
    "################",
    "################",
    "###..######..###",   # eyes
    "###..######..###",
    "################",
    "#######..#######",   # nose
    "################",
    "################",
    ".##############.",
    "..############..",
    "....########....",
    "................",
]

dog_drawing = [
    ".....######.....",
    "...##########...",
    "..############..",
    "################",
    "################",
    "###.########.###",   # floppy ears on both sides
    "###.#..##..#.###",   # eyes
    "###.#..##..#.###",
    "###.########.###",
    "###.###..###.###",   # nose
    "###.###..###.###",
    "###.########.###",
    "....########....",
    ".....######.....",
    "......####......",
    "................",
]

# Turn each drawing into numbers: 1 = black, 0 = white
def to_numbers(drawing):
    return np.array([[1 if pixel == '#' else 0 for pixel in row] for row in drawing])

cat = to_numbers(cat_drawing)
dog = to_numbers(dog_drawing)
print("Cat image shape:", cat.shape)
print("Format: ", type(cat))

In [ ]:
# Plot the two images
fig, axes = plt.subplots(1, 2, figsize = (8, 4))
axes[0].imshow(cat, cmap = 'gray_r')
axes[0].set_title('Typical cat')
axes[1].imshow(dog, cmap = 'gray_r')
axes[1].set_title('Typical dog')
plt.show()

In [ ]:
# The input of the model: the image written as one long vector of 256 numbers
cat_vector = cat.flatten()
dog_vector = dog.flatten()
print("Cat as a vector:", cat_vector)
print("Shape:", cat_vector.shape)

plt.figure(figsize = (14, 1.5))
plt.imshow(cat_vector.reshape(1, -1), cmap = 'gray_r')
plt.yticks([])
plt.xlabel('pixel number')
plt.title('The cat image as a vector of 256 pixels')
plt.show()

In [ ]:
# Labels as one-hot vectors: (cat, dog)
classes = ['cat', 'dog']
label_cat = np.array([1, 0])
label_dog = np.array([0, 1])
print("Label for a cat:", label_cat)
print("Label for a dog:", label_dog)

In [ ]:
# New images to classify: copies of the typical images with some pixels flipped at random (noise)
def add_noise(vector, n_pixels):
    noisy = vector.copy()
    flip = np.random.choice(len(vector), size = n_pixels, replace = False)
    noisy[flip] = 1 - noisy[flip]
    return noisy

np.random.seed(2)
images = [add_noise(cat_vector, 15), add_noise(dog_vector, 15), add_noise(cat_vector, 60), add_noise(cat_vector, 120)]
labels = [label_cat, label_dog, label_cat, label_cat]

fig, axes = plt.subplots(1, 4, figsize = (14, 4))
for i in range(4):
    axes[i].imshow(images[i].reshape(16, 16), cmap = 'gray_r')
    axes[i].set_title(f'Image {i}: a {classes[np.argmax(labels[i])]}')
plt.show()

In [ ]:
# Model: a very simple rule, not learned
# Look only at the pixels where a typical cat and a typical dog are different,
# and count how many of them agree with the cat and how many with the dog
different = cat_vector != dog_vector

def model(image):
    agree_cat = np.sum(image[different] == cat_vector[different])
    agree_dog = np.sum(image[different] == dog_vector[different])
    return np.array([agree_cat, agree_dog]) / (agree_cat + agree_dog)

print("Pixels where cat and dog differ:", np.sum(different))
print("Output for the typical cat:", model(cat_vector))

In [ ]:
# Loss: sum of squared differences between the label y and the prediction y_pred
def loss(y, y_pred):
    return np.sum((y - y_pred)**2)

In [ ]:
# Follow one image through the pipeline: input -> model -> output -> loss with the label
def show_pipeline(image, label):
    output = model(image)

    print(" class | label y | output y_pred | (y - y_pred)^2")
    print("-" * 48)
    for k in range(2):
        print(f"  {classes[k]}  |    {label[k]}    |     {output[k]:.2f}     |     {(label[k] - output[k])**2:.3f}")
    print("-" * 48)
    print(f"Loss (sum of the last column): {loss(label, output):.3f}")

    fig, axes = plt.subplots(1, 3, figsize = (15, 4))

    axes[0].imshow(image.reshape(16, 16), cmap = 'gray_r')
    axes[0].set_title('1. Input: image (256 pixels)')

    position = np.arange(2)
    axes[1].bar(position - 0.2, label, width = 0.4, color = 'green', label = 'Label y')
    axes[1].bar(position + 0.2, output, width = 0.4, color = 'red', label = 'Output y_pred')
    axes[1].set_xticks(position)
    axes[1].set_xticklabels(classes)
    axes[1].set_ylim(0, 1.1)
    axes[1].set_title('2. Output of the model vs label')
    axes[1].legend()

    axes[2].bar(classes, (label - output)**2, color = 'gray')
    axes[2].set_ylim(0, 1.1)
    axes[2].set_ylabel('(y - y_pred)^2')
    axes[2].set_title(f'3. Loss = sum of the bars = {loss(label, output):.3f}')

    plt.tight_layout()
    plt.show()

# Image 0: a cat with a little noise
show_pipeline(images[0], labels[0])

In [ ]:
# Image 1: a dog with a little noise
show_pipeline(images[1], labels[1])

In [ ]:
# Image 2: a cat with more noise, the model is less sure
show_pipeline(images[2], labels[2])

In [ ]:
# Image 3: a cat with a lot of noise, the model gets it wrong and the loss is large
show_pipeline(images[3], labels[3])

In [ ]:
# Summary of all images
print(" image | true class | output (cat, dog) | predicted | loss")
print("-" * 58)
for i in range(4):
    output = model(images[i])
    true_class = classes[np.argmax(labels[i])]
    predicted = classes[np.argmax(output)]
    print(f"   {i}   |    {true_class}     |   ({output[0]:.2f}, {output[1]:.2f})    |    {predicted}    | {loss(labels[i], output):.3f}")

In [ ]:
# Check the loss with the example from the maths: a cat with output (0.8, 0.2)
y = np.array([1, 0])
y_pred = np.array([0.8, 0.2])
by_hand = (1 - 0.8)**2 + (0 - 0.2)**2
print(f"By hand: {by_hand:.2f}")
print(f"numpy:   {loss(y, y_pred):.2f}")
print("Close: ", np.isclose(by_hand, loss(y, y_pred)))